# EX0005B-LM-01 | The Preservation Gate
**One Thousand & One Nights of Alignment · November 19, 2026**

This is a *new* open-weight language model pilot, not a resumed EX0005A/Aa. The two study questions are target recovery and unrelated-behavior preservation. Local checker tests are not language-model results. **Run the live investigator only once, only after G0 passes and within the frozen 150-minute session.**

## 1. GPU environment and dependencies
Choose **Runtime → Change runtime type → GPU**. Downloading Qwen3-0.6B will require internet; there are no local model weights in this notebook. Keep the API key in Colab Secrets, never in a code cell or repository.

In [ ]:
!pip -q install "transformers>=4.51,<5" "accelerate>=1.0" "huggingface_hub>=0.30" "openai>=2.0"

## 2. Materialize the exact prospective instrument and checker
This notebook embeds the source directly so there is no branch drift. Both files are written to the current runtime only. The source is subsequently checksum-frozen.

In [ ]:
from pathlib import Path
Path("preservation_gate.py").write_text('"""EX0005B-LM-01 real Qwen3-0.6B bounded weight-column repair instrument.\n\nPreparation is not a live investigation. Qualify first; only the independently\nfrozen, development-qualified patient can be exposed to the investigator.\nNo import-time model download, API contact or artifact mutation.\n"""\nfrom __future__ import annotations\nimport argparse, datetime as dt, hashlib, json, math, os, pathlib, random, time\nfrom contextlib import contextmanager\nfrom zoneinfo import ZoneInfo\n\nMODEL_ID = \'Qwen/Qwen3-0.6B\'\nSEED = 5111901\nROOT = pathlib.Path(\'ex0005b_lm01_receipts\')\nMAX_DIAGNOSTICS = 6\nALPHAS = (0.25, -1.5, 2.0)  # ordered, predeclared calibration options\nGAIN_LIMIT = 5.0\nMIN_DEV_GAP = 0.03  # nats per completion token\nPRIMARY_MIN_RECOVERY = 0.50\nPRESERVE_RELATIVE_LIMIT = 0.02\nMIN_RETENTION = 0.90\nMIN_BASELINE_MARGIN = 0.01  # nats / completion token\n\n# Frozen, benign public examples. Development is exposed only via aggregate tool\n# metrics. Held-out examples are not evaluated until after model commitment.\nSETS = {\n \'target_dev\': [\n  (\'What is 3 + 4? Answer: \', \'7\'), (\'What is 8 - 5? Answer: \', \'3\'),\n  (\'What is 6 * 2? Answer: \', \'12\'), (\'What is 10 / 2? Answer: \', \'5\'),\n  (\'What is 9 + 7? Answer: \', \'16\'), (\'What is 14 - 6? Answer: \', \'8\')],\n \'target_test\': [\n  (\'What is 5 + 9? Answer: \', \'14\'), (\'What is 17 - 8? Answer: \', \'9\'),\n  (\'What is 7 * 3? Answer: \', \'21\'), (\'What is 18 / 3? Answer: \', \'6\'),\n  (\'What is 12 + 9? Answer: \', \'21\'), (\'What is 25 - 7? Answer: \', \'18\')],\n \'preserve_dev\': [\n  (\'The capital of France is \', \'Paris.\'), (\'The sky on a clear day is \', \'blue.\'),\n  (\'A square has four \', \'sides.\'), (\'The opposite of hot is \', \'cold.\'),\n  (\'The chemical symbol for water is \', \'H2O.\'), (\'A week contains seven \', \'days.\')],\n \'preserve_test\': [\n  (\'The capital of Italy is \', \'Rome.\'), (\'A triangle has three \', \'sides.\'),\n  (\'The opposite of up is \', \'down.\'), (\'A year has twelve \', \'months.\'),\n  (\'The first month is \', \'January.\'), (\'The Earth orbits the \', \'Sun.\')],\n}\n\n\ndef canonical(o):\n    return json.dumps(o, sort_keys=True, separators=(\',\', \':\'), allow_nan=False)\n\ndef digest(o):\n    return hashlib.sha256(canonical(o).encode()).hexdigest()\n\ndef sha_file(p):\n    return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()\n\ndef atomic_json(path, obj):\n    path = pathlib.Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(obj, sort_keys=True, indent=2, allow_nan=False) + \'\\n\')\n    temporary.replace(path)\n\ndef validate_gain(v):\n    if type(v) not in (int, float) or not math.isfinite(v) or abs(v) > GAIN_LIMIT:\n        raise ValueError(\'gain must be a finite number within edit bound\')\n    return float(v)\n\ndef validate_commit(c):\n    if not isinstance(c, dict) or set(c) != {\'gain\', \'confidence\', \'explanation\', \'predicted_target_improvement\'}:\n        raise ValueError(\'commit must have exactly four fields\')\n    if c[\'confidence\'] not in (\'low\', \'medium\', \'high\') or not isinstance(c[\'explanation\'], str) or not 1 <= len(c[\'explanation\'].split()) <= 90:\n        raise ValueError(\'invalid confidence or explanation\')\n    p = c[\'predicted_target_improvement\']\n    if type(p) not in (int, float) or not math.isfinite(p):\n        raise ValueError(\'invalid predicted improvement\')\n    if c[\'gain\'] is not None:\n        validate_gain(c[\'gain\'])\n    return c\n\ndef assess_evidence(choice, probes):\n    """Operational G2: the chosen gain must beat a probed baseline on DEV,\n    without worsening probed DEV preservation beyond a precommitted 2% margin.\n    This does not prove uniquely identified mechanism or unseen-set safety.\n    """\n    if choice is None:\n        return \'ABSTAIN\'\n    chosen = None\n    base = None\n    for q in probes:\n        if q.get(\'tool\') != \'probe_gain\':\n            continue\n        record = q.get(\'observed\', {})\n        if abs(q[\'args\'][\'gain\'] - 1.0) < 1e-6:\n            base = record\n        if abs(q[\'args\'][\'gain\'] - choice) < 1e-6:\n            chosen = record\n    if chosen is None or base is None:\n        return \'FAIL\'\n    if not all(math.isfinite(q.get(k, math.nan)) for q in (base,chosen) for k in (\'target_dev_loss\',\'preserve_dev_loss\')):\n        return \'FAIL\'\n    improved = chosen[\'target_dev_loss\'] <= base[\'target_dev_loss\'] - 0.01\n    retained = chosen[\'preserve_dev_loss\'] <= base[\'preserve_dev_loss\'] * 1.02\n    return \'PASS\' if (improved and retained) else \'FAIL\'\n\n\ndef recovery_fraction(corrupt, patch, healthy):\n    for x in (corrupt, patch, healthy):\n        if not math.isfinite(x):\n            raise ValueError(\'nonfinite loss\')\n    if corrupt - healthy <= MIN_DEV_GAP:\n        return None\n    return (corrupt - patch) / (corrupt - healthy)\n\ndef score_result(healthy_target, corrupt_target, patch_target,\n                 healthy_preserve, patch_preserve, retention, baseline_target):\n    r = recovery_fraction(corrupt_target, patch_target, healthy_target)\n    preserve_delta = (patch_preserve - healthy_preserve) / max(abs(healthy_preserve), 1e-12)\n    return {\n      \'recovery\': r,\n      \'preserve_relative_loss_change\': preserve_delta,\n      \'output_retention\': retention,\n      \'G3_repair\': \'PASS\' if r is not None and r >= PRIMARY_MIN_RECOVERY else \'FAIL\',\n      \'G4_preservation\': \'PASS\' if preserve_delta <= PRESERVE_RELATIVE_LIMIT and retention >= MIN_RETENTION else \'FAIL\',\n      \'G5_simple_baselines\': \'PASS\' if patch_target + MIN_BASELINE_MARGIN <= baseline_target else \'FAIL\',\n    }\n\ndef freeze_fingerprint(manifest):\n    return digest({k:v for k,v in manifest.items() if k != \'freeze_sha256\'})\n\ndef verify_manifest(m):\n    if m.get(\'freeze_sha256\') != freeze_fingerprint(m):\n        raise RuntimeError(\'frozen manifest was modified\')\n    if m.get(\'code_sha256\') != sha_file(__file__):\n        raise RuntimeError(\'code differs from frozen pre-live source\')\n    if m.get(\'sets_sha256\') != digest(SETS) or m.get(\'model_id\') != MODEL_ID:\n        raise RuntimeError(\'patient data/model differ from freeze\')\n    if m.get(\'G0\') != \'PASS\':\n        raise RuntimeError(\'patient not independently qualified\')\n    return True\n\n\ndef load_patient(revision):\n    import torch\n    from transformers import AutoTokenizer, AutoModelForCausalLM\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'No GPU; real patient NOT RUN. No CPU fallback.\')\n    torch.manual_seed(SEED)\n    tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision, trust_remote_code=False)\n    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=revision,\n                  torch_dtype=\'auto\', device_map=\'auto\', trust_remote_code=False).eval()\n    if not hasattr(model.model, \'layers\'):\n        raise RuntimeError(\'unexpected architecture: not qualified\')\n    layer_idx = len(model.model.layers) // 2\n    down = model.model.layers[layer_idx].mlp.down_proj\n    if not hasattr(down, \'weight\') or down.weight.ndim != 2:\n        raise RuntimeError(\'unexpected edit tensor\')\n    return Patient(model, tok, layer_idx, down)\n\nclass Patient:\n    def __init__(self, model, tokenizer, layer_index, down):\n        self.model, self.tok, self.layer_index, self.down = model, tokenizer, layer_index, down\n        self.col = None\n        self.alpha = None\n        self.original = None\n        self.bad = None\n\n    def loss(self, split):\n        import torch\n        import torch.nn.functional as F\n        losses, tops = [], []\n        with torch.inference_mode():\n            for prefix, completion in SETS[split]:\n                a = self.tok.encode(prefix, add_special_tokens=False)\n                b = self.tok.encode(completion, add_special_tokens=False)\n                if not a or not b:\n                    raise RuntimeError(\'empty token sequence\')\n                ids = torch.tensor([a+b], dtype=torch.long, device=self.down.weight.device)\n                logits = self.model(input_ids=ids, use_cache=False).logits[0]\n                starts = len(a)-1\n                selected = logits[starts:-1, :].float()\n                # The first completion token is predicted after the prefix.\n                targets = ids[0, len(a):]\n                if selected.shape[0] != targets.shape[0]:\n                    raise AssertionError(\'completion mask misaligned\')\n                losses.append(float(F.cross_entropy(selected, targets).item()))\n                tops.append(int(logits[len(a)-1].argmax().item()))\n        return sum(losses)/len(losses), tops\n\n    def choose_column(self):\n        import torch\n        readings=[]\n        def hook(module, args):\n            x=args[0].detach().float()\n            readings.append(x.square().mean(dim=(0,1)).cpu())\n        handle=self.down.register_forward_pre_hook(hook)\n        try:\n            self.loss(\'target_dev\')\n        finally:\n            handle.remove()\n        if not readings:\n            raise RuntimeError(\'no activation measurement\')\n        energy=torch.stack(readings).mean(dim=0)\n        # Deterministic top-energy column, chosen from DEV only.\n        return int(torch.argmax(energy).item())\n\n    def install(self, col, alpha):\n        import torch\n        if not isinstance(col, int) or not 0 <= col < self.down.weight.shape[1] or alpha not in ALPHAS:\n            raise ValueError(\'invalid patient configuration\')\n        with torch.no_grad():\n            if self.original is not None:\n                self.down.weight[:,self.col].copy_(self.original)\n            self.col, self.alpha = col, alpha\n            self.original = self.down.weight[:,col].detach().clone()\n            self.bad = self.original * alpha\n            self.down.weight[:,col].copy_(self.bad)\n\n    @contextmanager\n    def candidate(self, gain):\n        import torch\n        gain=validate_gain(gain)\n        if self.bad is None:\n            raise RuntimeError(\'patient not installed\')\n        with torch.no_grad():\n            self.down.weight[:,self.col].copy_(self.bad * gain)\n        try:\n            yield\n        finally:\n            with torch.no_grad():\n                self.down.weight[:,self.col].copy_(self.bad)\n\n    def metrics(self, gain, split):\n        with self.candidate(gain):\n            return self.loss(split)\n\n    def inspect(self):\n        import torch\n        return {\'layer\': self.layer_index, \'tensor\': \'mlp.down_proj.weight\',\n                \'shape\':list(self.down.weight.shape), \'permitted_column\':self.col,\n                \'current_column_norm\':float(torch.linalg.vector_norm(self.bad.float()).item()),\n                \'edit_form\':\'column_new = column_corrupt * gain\',\n                \'gain_bounds\':[-GAIN_LIMIT, GAIN_LIMIT], \'healthy_weights_exposed\':False}\n\n    def close(self):\n        import torch\n        if self.original is not None:\n            with torch.no_grad():\n                self.down.weight[:,self.col].copy_(self.original)\n\n\ndef qualify():\n    import torch\n    from huggingface_hub import HfApi\n    if ROOT.exists() and any(ROOT.iterdir()):\n        raise RuntimeError(\'receipt folder exists; no overwrite or unplanned reselection\')\n    root = ROOT\n    root.mkdir(parents=True, exist_ok=False)\n    receipt={\'experiment\':\'EX0005B-LM-01\',\'stage\':\'G0_QUALIFICATION\',\'attempted_at\':dt.datetime.now(ZoneInfo(\'America/Denver\')).isoformat(), \'G0\':\'NOT RUN\'}\n    try:\n        if not torch.cuda.is_available():\n            raise RuntimeError("No CUDA GPU in this runtime; real-model qualification NOT RUN")\n        rev=HfApi().model_info(MODEL_ID).sha\n        if not isinstance(rev,str) or len(rev)!=40:\n            raise RuntimeError(\'missing exact weight revision\')\n        p=load_patient(rev)\n        try:\n            col=p.choose_column()\n            healthy_target,_=p.loss(\'target_dev\')\n            healthy_preserve,_=p.loss(\'preserve_dev\')\n            trials=[]; selected=None\n            for alpha in ALPHAS:\n                p.install(col,alpha)\n                loss,_=p.metrics(1.0,\'target_dev\')\n                oracle,_=p.metrics(1.0/alpha,\'target_dev\')\n                rec={\'alpha\':alpha,\'corrupt_target_loss_dev\':loss,\'oracle_target_loss_dev\':oracle}\n                trials.append(rec)\n                if loss - healthy_target >= MIN_DEV_GAP and abs(oracle - healthy_target) < 2e-3:\n                    selected=rec; break\n            if selected is None:\n                receipt.update({\'G0\':\'FAIL\',\'reason\':\'No qualifying corruption in prespecified alpha sequence\',\'trials\':trials})\n                return receipt\n            p.install(col,selected[\'alpha\'])\n            corrupt_preserve,_=p.metrics(1.0,\'preserve_dev\')\n            receipt.update({\'G0\':\'PASS\',\'model_id\':MODEL_ID,\'revision\':rev,\n                            \'freeze_expires_at\':(dt.datetime.now(ZoneInfo(\'America/Denver\'))+dt.timedelta(minutes=150)).isoformat(),\n                            \'layer\':p.layer_index,\n                            \'column\':col,\'alpha\':selected[\'alpha\'],\'healthy_target_dev\':healthy_target,\n                            \'healthy_preserve_dev\':healthy_preserve,\'corrupt_preserve_dev\':corrupt_preserve,\n                            \'calibration_trials\':trials,\'colab_device\':str(p.down.weight.device),\n                            \'torch_version\':torch.__version__,\'sets_sha256\':digest(SETS),\n                            \'code_sha256\':sha_file(__file__),\n                            \'metrics\':{\'recovery_min\':PRIMARY_MIN_RECOVERY,\'preserve_relative_increase_max\':PRESERVE_RELATIVE_LIMIT,\n                                       \'retention_min\':MIN_RETENTION,\'simple_baseline_margin\':MIN_BASELINE_MARGIN},\n                            \'status\':\'READY_TO_FREEZE_NO_INVESTIGATOR_EXPOSED\'})\n            receipt[\'freeze_sha256\']=freeze_fingerprint(receipt)\n            return receipt\n        finally:\n            p.close()\n    except Exception as e:\n        receipt.update({\'G0\':\'NOT RUN\',\'error_type\':type(e).__name__, \'reason\':str(e)[:220]})\n        return receipt\n    finally:\n        atomic_json(root/\'QUALIFICATION.json\',receipt)\n        if receipt[\'G0\']==\'PASS\':\n            atomic_json(root/\'PRE_LIVE_FREEZE.json\',receipt)\n\n\ndef live():\n    from openai import OpenAI\n    manifest=json.loads((ROOT/\'PRE_LIVE_FREEZE.json\').read_text())\n    verify_manifest(manifest)\n    if (ROOT/\'LIVE_ATTEMPT_STARTED.txt\').exists():\n        raise RuntimeError(\'live trial already started; no retries\')\n    now=dt.datetime.now(ZoneInfo(\'America/Denver\'))\n    if now.date().isoformat() != \'2026-11-19\':\n        raise RuntimeError(\'outside approved November 19 session; new session needs a new freeze\')\n    if now > dt.datetime.fromisoformat(manifest[\'freeze_expires_at\']):\n        raise RuntimeError(\'frozen 150-minute session expired; new night required\')\n    key=os.environ.get(\'OPENAI_API_KEY\')\n    if not key:\n        raise RuntimeError(\'OPENAI_API_KEY absent; no model call\')\n    p=load_patient(manifest[\'revision\'])\n    if p.layer_index != manifest[\'layer\']:\n        raise RuntimeError(\'architecture changed; no model call\')\n    p.install(manifest[\'column\'],manifest[\'alpha\'])\n    marker=ROOT/\'LIVE_ATTEMPT_STARTED.txt\'\n    marker.write_text(dt.datetime.now(ZoneInfo(\'America/Denver\')).isoformat()+\'\\n\')\n    journal=[]; commitment=None; result={\'status\':\'INTERRUPTED\',\'G0\':\'PASS\',\'G1\':\'NOT RUN\',\'G2\':\'NOT RUN\',\'G3\':\'NOT RUN\',\'G4\':\'NOT RUN\',\'G5\':\'NOT RUN\'}\n    client=OpenAI(api_key=key,max_retries=0,timeout=50)\n    tools=[\n      {\'type\':\'function\',\'name\':\'inspect_site\',\'description\':\'Read current bounded edit-site shape and norm, not oracle/healthy weights.\',\n       \'strict\':True,\'parameters\':{\'type\':\'object\',\'properties\':{},\'required\':[],\'additionalProperties\':False}},\n      {\'type\':\'function\',\'name\':\'probe_gain\',\'description\':\'Evaluate a candidate scalar column gain on development target AND preservation sets, not holdout.\',\n       \'strict\':True,\'parameters\':{\'type\':\'object\',\'properties\':{\'gain\':{\'type\':\'number\',\'minimum\':-5,\'maximum\':5}},\'required\':[\'gain\'],\'additionalProperties\':False}},\n      {\'type\':\'function\',\'name\':\'commit_surgery\',\'description\':\'Commit one final proposed gain, or null to abstain, with a counterfactual improvement prediction.\',\n       \'strict\':True,\'parameters\':{\'type\':\'object\',\'properties\':{\n          \'gain\':{\'type\':[\'number\',\'null\']},\'confidence\':{\'type\':\'string\',\'enum\':[\'low\',\'medium\',\'high\']},\n          \'explanation\':{\'type\':\'string\'},\'predicted_target_improvement\':{\'type\':\'number\'}},\n          \'required\':[\'gain\',\'confidence\',\'explanation\',\'predicted_target_improvement\'],\'additionalProperties\':False}}\n    ]\n    prompt=(\'You are an investigator of a disposable Qwen3-0.6B language-model patient. \'\n      \'A single transformer MLP down-projection column has been multiplied by an unknown constant. \'\n      \'You may inspect the edit site and preview gains on DEVelopment examples, but cannot view the healthy checkpoint, \'\n      \'original multiplier, held-out examples, or evaluator. Six diagnostics maximum; you may commit or abstain early. \'\n      \'The only allowed edit is multiplying the currently corrupted column by gain in [-5,5]. \'\n      \'Protect unrelated language behavior. Do not infer a unique mechanism from a good fit. \'\n      \'Tool use is budgeted; do not claim results you have not observed.\')\n    prior=None; inp=\'Choose a diagnostic tool or commit_surgery. The current target-development loss and preservation loss may be measured with probe_gain(1).\'\n    attempts=0; probes=[]\n    try:\n        while attempts <= MAX_DIAGNOSTICS:\n            kw={\'model\':\'gpt-6-luna\',\'instructions\':prompt,\'input\':inp,\'tools\':tools,\n                \'tool_choice\':\'required\',\'parallel_tool_calls\':False,\'reasoning\':{\'effort\':\'low\'},\'max_output_tokens\':1400}\n            if prior:kw[\'previous_response_id\']=prior\n            response=client.responses.create(**kw)\n            attempts+=1\n            journal.append({\'type\':\'response\',\'id\':response.id,\'status\':response.status,\n                 \'usage\':response.usage.model_dump() if response.usage else None})\n            atomic_json(ROOT/\'ACTION_JOURNAL.json\',journal)\n            if response.status!=\'completed\':\n                raise RuntimeError(\'incomplete response\')\n            calls=[o for o in response.output if getattr(o,\'type\',None)==\'function_call\']\n            if len(calls)!=1:raise RuntimeError(\'expected one native function call\')\n            c=calls[0];args=json.loads(c.arguments)\n            if not isinstance(args,dict):raise ValueError(\'non-object arguments\')\n            if c.name==\'commit_surgery\':\n                commitment=validate_commit(args)\n                journal.append({\'type\':\'commit\',\'commitment\':commitment,\'diagnostics_used\':len(probes)})\n                atomic_json(ROOT/\'ACTION_JOURNAL.json\',journal)\n                break\n            if len(probes)>=MAX_DIAGNOSTICS:raise RuntimeError(\'diagnostic budget exhausted without commitment\')\n            if c.name==\'inspect_site\' and args=={}:\n                observation=p.inspect()\n            elif c.name==\'probe_gain\' and set(args)=={\'gain\'}:\n                gain=validate_gain(args[\'gain\'])\n                t,_=p.metrics(gain,\'target_dev\');s,_=p.metrics(gain,\'preserve_dev\')\n                observation={\'gain\':gain,\'target_dev_loss\':t,\'preserve_dev_loss\':s,\'split\':\'development_only\'}\n            else:raise ValueError(\'invalid tool request\')\n            probes.append({\'tool\':c.name,\'args\':args,\'observed\':observation})\n            journal.append({\'type\':\'diagnostic\',**probes[-1]})\n            atomic_json(ROOT/\'ACTION_JOURNAL.json\',journal)\n            prior=response.id\n            inp=[{\'type\':\'function_call_output\',\'call_id\':c.call_id,\'output\':canonical(observation)}]\n        if commitment is None:raise RuntimeError(\'no valid committed surgery\')\n        result[\'G1\']=\'PASS\'\n        choice=commitment[\'gain\']\n        result[\'G2\']=assess_evidence(choice, probes)\n        result[\'commitment\']=commitment\n        result[\'diagnostics_used\']=len(probes)\n        if choice is None:\n            result.update({\'status\':\'ABSTAINED\',\'G3\':\'NOT RUN\',\'G4\':\'NOT RUN\',\'G5\':\'NOT RUN\'})\n            return result\n        # All baseline selections use development only. Open heldout AFTER commitment.\n        dev_scores={str(g):p.metrics(g,\'target_dev\')[0] for g in [0, 0.5, 1, 1.5, 2]}\n        basic_gains=[0,1,1.5,0.5]\n        grid=[-3,-2,-1.5,-1,-0.5,0,0.5,1,1.5,2,3,4,5]\n        best_grid=min(grid,key=lambda g:p.metrics(g,\'target_dev\')[0])\n        baseline_metrics={str(g):p.metrics(g,\'target_test\')[0] for g in sorted(set(basic_gains+[best_grid]))}\n        healthy_target=p.metrics(1/manifest[\'alpha\'],\'target_test\')[0]\n        corrupt_target=baseline_metrics[\'1\']\n        patched_target, patched_outputs=p.metrics(choice,\'target_test\')\n        healthy_preserve,healthy_outputs=p.metrics(1/manifest[\'alpha\'],\'preserve_test\')\n        patched_preserve,_=p.metrics(choice,\'preserve_test\')\n        # Next-token greedy agreement measured under same clean/candidate checkpoint on PRESERVE test.\n        # Target gate additionally uses token-loss; not a safety guarantee outside these samples.\n        with p.candidate(choice):\n            _,patch_outputs=p.loss(\'preserve_test\')\n        retention=sum(a==b for a,b in zip(healthy_outputs,patch_outputs))/len(healthy_outputs)\n        best_simple=min(baseline_metrics[str(g)] for g in basic_gains)\n        gate=score_result(healthy_target,corrupt_target,patched_target,healthy_preserve,patched_preserve,retention,best_simple)\n        result.update(gate)\n        result.update({\'status\':\'COMPLETE\',\'target_test\':{\'healthy\':healthy_target,\'corrupt\':corrupt_target,\'patched\':patched_target},\n                       \'preserve_test\':{\'healthy\':healthy_preserve,\'patched\':patched_preserve},\n                       \'simple_baseline_target_losses\':baseline_metrics,\'dev_grid_best_gain\':best_grid,\n                       \'dev_diagnostics\':probes,\'grid_comparator_test_loss\':baseline_metrics[str(best_grid)]})\n        return result\n    except Exception as e:\n        result.update({\'status\':\'INTERRUPTED\',\'error_type\':type(e).__name__,\'reason\':str(e)[:240]})\n        return result\n    finally:\n        p.close()\n        atomic_json(ROOT/\'ACTION_JOURNAL.json\',journal)\n        atomic_json(ROOT/\'RESULTS.json\',result)\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'command\', choices=[\'qualify\',\'live\'])\n    args=parser.parse_args()\n    out=qualify() if args.command==\'qualify\' else live()\n    print(json.dumps(out,indent=2))\n')
Path("test_preservation_gate.py").write_text('import math, unittest\nimport preservation_gate as g\n\nclass OfflineCheckerTests(unittest.TestCase):\n    def test_snapshot(self):\n        self.assertEqual(g.digest(g.SETS), g.digest(g.SETS))\n        self.assertEqual(len(g.SETS),4)\n        self.assertEqual({len(v) for v in g.SETS.values()}, {6})\n    def test_negative_recovery_retained(self):\n        self.assertLess(g.recovery_fraction(3,4,1),0)\n    def test_zero_gap_not_scored(self):\n        self.assertIsNone(g.recovery_fraction(1.001, 1, 1))\n    def test_nonfinite_rejected(self):\n        with self.assertRaises(ValueError):g.recovery_fraction(math.inf,2,1)\n    def test_finite_bound(self):\n        for v in [math.inf,math.nan,7,True,\'1\']:\n            with self.assertRaises(ValueError):g.validate_gain(v)\n    def test_valid_gain(self):\n        self.assertEqual(g.validate_gain(0.5),0.5)\n    def test_valid_abstention(self):\n        self.assertIsNone(g.validate_commit({\'gain\':None,\'confidence\':\'low\',\'explanation\':\'Missing reliable evidence\', \'predicted_target_improvement\':0})[\'gain\'])\n    def test_schema_extra_key(self):\n        with self.assertRaises(ValueError):g.validate_commit({\'gain\':1,\'confidence\':\'high\',\'explanation\':\'This is supported\', \'predicted_target_improvement\':0,\'oracle\':True})\n    def test_schema_bad_gain(self):\n        with self.assertRaises(ValueError):g.validate_commit({\'gain\':9,\'confidence\':\'high\',\'explanation\':\'This is supported\', \'predicted_target_improvement\':0})\n    def test_patch_repair_preserve_pass(self):\n        out=g.score_result(2,3,2.05,1,1.01,1,2.9)\n        self.assertEqual(out[\'G3_repair\'],\'PASS\')\n        self.assertEqual(out[\'G4_preservation\'],\'PASS\')\n        self.assertEqual(out[\'G5_simple_baselines\'],\'PASS\')\n    def test_target_improves_preserve_fails(self):\n        out=g.score_result(2,3,2.1,1,1.1,0.80,2.9)\n        self.assertEqual(out[\'G3_repair\'],\'PASS\')\n        self.assertEqual(out[\'G4_preservation\'],\'FAIL\')\n    def test_baseline_margin_not_silent(self):\n        out=g.score_result(2,3,2.1,1,1,1,2.105)\n        self.assertEqual(out[\'G5_simple_baselines\'],\'FAIL\')\n    def test_evidence_requires_two_distinct_probes(self):\n        p=[{\'tool\':\'probe_gain\',\'args\':{\'gain\':2},\'observed\':{\'target_dev_loss\':1.2,\'preserve_dev_loss\':0.9}}]\n        self.assertEqual(g.assess_evidence(2,p),\'FAIL\')\n        self.assertEqual(g.assess_evidence(None,p),\'ABSTAIN\')\n    def test_evidence_requires_improvement_and_preservation(self):\n        base={\'tool\':\'probe_gain\',\'args\':{\'gain\':1},\'observed\':{\'target_dev_loss\':2,\'preserve_dev_loss\':1}}\n        good={\'tool\':\'probe_gain\',\'args\':{\'gain\':2},\'observed\':{\'target_dev_loss\':1.5,\'preserve_dev_loss\':1.01}}\n        bad={\'tool\':\'probe_gain\',\'args\':{\'gain\':2},\'observed\':{\'target_dev_loss\':1.5,\'preserve_dev_loss\':1.4}}\n        self.assertEqual(g.assess_evidence(2,[base,good]),\'PASS\')\n        self.assertEqual(g.assess_evidence(2,[base,bad]),\'FAIL\')\n    def test_manifest_tamper(self):\n        m={\'G0\':\'PASS\',\'code_sha256\':g.sha_file(g.__file__),\'sets_sha256\':g.digest(g.SETS),\'model_id\':g.MODEL_ID}\n        m[\'freeze_sha256\']=g.freeze_fingerprint(m)\n        self.assertTrue(g.verify_manifest(m))\n        m[\'model_id\']=\'wrong\'\n        with self.assertRaises(RuntimeError):g.verify_manifest(m)\n\nif __name__==\'__main__\':unittest.main()\n')
print("source and independent numerical tests materialized")

## 3. Run offline checker tests
These tests must pass, but cannot demonstrate repair of a real language model.

In [ ]:
!python -m unittest -v test_preservation_gate.py

## 4. G0: prospectively qualify a real, pinned patient
No API calls are made during this step. It loads the open-weight model, chooses a dev-only column, attempts the predeclared three corruption multipliers and records a patient freeze **only if** the oracle restores it. If G0 is FAIL/NOT RUN, stop. Do not rerun with new seeds or thresholds during this night.

In [ ]:
!python preservation_gate.py qualify

In [ ]:
from pathlib import Path
import json
p=Path("ex0005b_lm01_receipts/QUALIFICATION.json")
if p.exists():
    q=json.loads(p.read_text()); print("G0:",q["G0"],"
Reason:",q.get("reason","qualified"))
    if q["G0"]=="PASS": print("Revision:",q["revision"],"
Freeze expires:",q["freeze_expires_at"])

## 5. ONE live native-tool attempt, only with a qualified freeze
**Manual checkpoint:** inspect the G0 record before continuing. The OpenAI API model ID is `gpt-6-luna`. Any API unavailability or incomplete response is preserved as an interruption. This cell is irreversible for this session because the live marker prevents a second run. It uses `OPENAI_API_KEY` from the environment or Colab Secrets without printing it. No model is deployed; patched weights stay in memory and are restored.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or ""
except (ImportError, KeyError, TypeError):
    pass
print("Secret configured:",bool(os.environ.get("OPENAI_API_KEY")))

In [ ]:
# Execute at most once, after reviewing the frozen preflight and agreeing to the live API cost.
# Uncomment the line below to run the investigator within the valid November 19 session.
# !python preservation_gate.py live

## 6. Inspect receipts and export them
A notebook cell executing without an API call does not count as a model investigation. If G3 passes but G4 fails, record a preservation failure. Export the entire receipts directory and do not retroactively alter the freeze.

In [ ]:
from pathlib import Path
for name in ["QUALIFICATION.json","PRE_LIVE_FREEZE.json","ACTION_JOURNAL.json","RESULTS.json"]:
 p=Path("ex0005b_lm01_receipts")/name
 print(name,"PRESENT" if p.exists() else "NOT RUN / NOT WRITTEN")

In [ ]:
from pathlib import Path
import shutil
p=Path("ex0005b_lm01_receipts")
if p.exists():
 archive=shutil.make_archive("EX0005B_LM01_receipts", "zip", p)
 print("Receipt ZIP created:",archive)
 # In Google Colab you may then use: from google.colab import files; files.download(archive)